# SCRFD-10G + Kopfdetektor: Gesichter und Köpfe in Patientenvideos verpixeln

**Ablauf:** Video lesen → Gesichter erkennen (SCRFD, ONNX) → Spuren bilden → Lücken füllen → glätten → Doppelte entfernen → nur die Gesichtsfläche verpixeln → Video schreiben.

Die Roh-Erkennungen werden einmal berechnet und gespeichert (`FACES_FILE`). Danach lassen sich Tracking und Rendern schnell wiederholen.

**Datenschutz:** Vor jedem Commit alle Ausgaben löschen. Rohdaten bleiben in `data/raw` (gitignored).

**Erweiterung (Abschnitte 12–15):** Zusätzlich zum Gesichtsdetektor läuft ein Kopfdetektor (YOLO11s). Er findet auch gedrehte oder verdeckte Köpfe. Beide Ergebnisse werden zusammengeführt und zusammen verpixelt.

## 1. Setup

In [ ]:
import pickle
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort
import torch
from torch.utils.data import IterableDataset

print("Python:", sys.executable)   # sollte auf .venv\Scripts\python.exe enden
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)

# GPU, falls vorhanden, sonst CPU (die Erkennung selbst läuft über onnxruntime auf der CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Rechengerät:", DEVICE)

PROJECT_ROOT = Path.cwd().parent
MODEL_SCRFD = PROJECT_ROOT / "models" / "buffalo_l" / "buffalo_l" / "det_10g.onnx"
INPUT_VIDEO = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
FACES_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_faces_1280.pkl"   # Roh-Erkennungen
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "PAT_1"

print("Modell gefunden:", MODEL_SCRFD.exists())
print("Eingabevideo gefunden:", INPUT_VIDEO.exists())

## 2. Video als PyTorch-Datenquelle

Wird von der Endkette nicht benutzt (dort liest OpenCV direkt). Bleibt für spätere Batch-Verarbeitung/GPU.

In [ ]:
class VideoFrames(IterableDataset):
    """Liest ein Video Frame für Frame und liefert (Frame-Nummer, Tensor)."""

    def __init__(self, path, max_frames=None):
        self.path = str(path)
        self.max_frames = max_frames

    def __iter__(self):
        cap = cv2.VideoCapture(self.path)
        try:
            i = 0
            while self.max_frames is None or i < self.max_frames:
                ok, frame = cap.read()
                if not ok:
                    break
                yield i, torch.from_numpy(frame).permute(2, 0, 1)
                i += 1
        finally:
            cap.release()

## 3. Der SCRFD-Detektor (ONNX)

In [ ]:
class SCRFD:
    def __init__(self, model_path, input_size=(1280, 736), score_threshold=0.5, nms_threshold=0.4, pad_value=127):
        self.pad_value = pad_value
        self.in_w, self.in_h = input_size
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.strides = [8, 16, 32]
        self.num_anchors = 2

        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print("Ausgaben des Netzes:", len(self.output_names), "(erwartet 9)")

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]

        # 1. Letterbox
        scale = min(self.in_w / iw, self.in_h / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        canvas = np.full((self.in_h, self.in_w, 3), self.pad_value, dtype=np.uint8)
        canvas[:new_h, :new_w] = cv2.resize(img_bgr, (new_w, new_h))

        # 2. Netz: (Pixel - 127.5) / 128, BGR -> RGB
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (self.in_w, self.in_h),
                                     (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})

        boxes, scores = [], []
        for idx, stride in enumerate(self.strides):
            s = outs[idx].reshape(-1)                      # Scores
            d = outs[idx + 3].reshape(-1, 4) * stride      # Abstände in Pixeln

            # 3. Gitterpunkte dieses Maßstabs, je 2 Anker
            gh, gw = self.in_h // stride, self.in_w // stride
            ys, xs = np.mgrid[:gh, :gw]
            centers = np.stack([xs, ys], axis=-1).astype(np.float32).reshape(-1, 2) * stride
            centers = np.repeat(centers, self.num_anchors, axis=0)

            for k in np.where(s >= self.score_threshold)[0]:
                cx, cy = centers[k]
                x1, y1 = cx - d[k, 0], cy - d[k, 1]
                x2, y2 = cx + d[k, 2], cy + d[k, 3]
                boxes.append([x1 / scale, y1 / scale, (x2 - x1) / scale, (y2 - y1) / scale])
                scores.append(float(s[k]))

        if not boxes:
            return 0, np.empty((0, 5), dtype=np.float32)

        # 4. NMS
        idxs = np.array(cv2.dnn.NMSBoxes(boxes, scores, self.score_threshold, self.nms_threshold)).reshape(-1)
        faces = np.array([[*boxes[i], scores[i]] for i in idxs], dtype=np.float32)
        return len(faces), faces


detector = SCRFD(MODEL_SCRFD, input_size=(1280, 736), score_threshold=0.5)

## 4. Hilfsfunktionen für Boxen

`iou`: Überlappung zweier Boxen (0 bis 1). `inter_over_smaller`: Anteil der kleineren Box, der in der anderen liegt.

In [ ]:
def iou(a, b):
    """Überlappung zweier Boxen (x, y, w, h): 0 = keine, 1 = identisch."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def inter_over_smaller(a, b):
    """Anteil der kleineren Box, der in der anderen liegt (0 bis 1)."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / max(1e-9, min(aw * ah, bw * bh))

## 5. Spuren bilden (Tracking)

Start ab Score 0,5, Weiterführen ab 0,2, Geschwindigkeit mit Dämpfung, Halten bis `max_age` Frames, schwache Nachbarn über `near_threshold`.

In [ ]:
def near_track(box, tracks, pad=0.5, min_frac=0.3):
    """True, wenn mindestens min_frac der Box im verpixelten Bereich einer Spur liegt."""
    x, y, w, h = box
    for t in tracks:
        tx, ty, tw, th = t["box"]
        rx0, ry0 = tx - pad * tw, ty - pad * th
        rx1, ry1 = tx + tw + pad * tw, ty + th + pad * th
        ix = max(0, min(x + w, rx1) - max(x, rx0))
        iy = max(0, min(y + h, ry1) - max(y, ry0))
        if ix * iy >= min_frac * w * h:
            return True
    return False


def track_with_ids(all_faces, start_threshold=0.5, low_threshold=0.2, match_iou=0.3,
                   max_age=8, damping=0.85, near_threshold=None):
    tracks, result, next_id = [], [], 0
    for faces in all_faces:
        cands = [(tuple(float(v) for v in f[:4]), float(f[4])) for f in faces if f[4] >= low_threshold]

        for t in tracks:                                    # erwartete Position
            x, y, w, h = t["box"]
            t["pred"] = (x + t["v"][0], y + t["v"][1], w, h)

        pairs = sorted(((iou(t["pred"], box), ti, ci)
                        for ti, t in enumerate(tracks) for ci, (box, s) in enumerate(cands)),
                       reverse=True)
        used_t, used_c = set(), set()
        for ov, ti, ci in pairs:
            if ov < match_iou:
                break
            if ti in used_t or ci in used_c:
                continue
            t, new = tracks[ti], cands[ci][0]
            steps = t["age"] + 1
            mvx = (new[0] - t["det"][0]) / steps
            mvy = (new[1] - t["det"][1]) / steps
            t["v"] = (0.5 * t["v"][0] + 0.5 * mvx, 0.5 * t["v"][1] + 0.5 * mvy)
            t["box"], t["det"], t["age"] = new, new, 0
            used_t.add(ti)
            used_c.add(ci)

        for ti, t in enumerate(tracks):                     # nicht gefunden: weiterlaufen
            if ti not in used_t:
                t["box"] = t["pred"]
                t["v"] = (t["v"][0] * damping, t["v"][1] * damping)
                t["age"] += 1
        tracks = [t for t in tracks if t["age"] <= max_age]

        for ci, (box, s) in enumerate(cands):               # neue Spuren
            if ci in used_c:
                continue
            if s < start_threshold:
                # NEU: schwacher Fund nur, wenn er an einer bestehenden Spur liegt
                if near_threshold is None or s < near_threshold or not near_track(box, tracks):
                    continue
            if any(inter_over_smaller(box, t["box"]) >= 0.5 for t in tracks):
                continue
            tracks.append({"id": next_id, "box": box, "det": box, "v": (0.0, 0.0), "age": 0})
            next_id += 1

        result.append([(t["id"], t["box"]) for t in tracks])
    return result

## 6. Lücken füllen (nur offline)

Linear zwischen dem Ende einer Spur und dem Anfang einer nahen neuen Spur.

In [ ]:
def bridge_gaps(tracked, gap_max=40, dist_factor=1.5):
    n = len(tracked)
    per_id = {}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box
    first = {tid: min(d) for tid, d in per_id.items()}
    last = {tid: max(d) for tid, d in per_id.items()}

    def center(b):
        return (b[0] + b[2] / 2, b[1] + b[3] / 2)

    added = [[] for _ in range(n)]
    for b_id in sorted(per_id, key=lambda t: first[t]):
        fb = first[b_id]
        boxb = per_id[b_id][fb]
        best, best_dist = None, None
        for a_id in per_id:                                  # früher beendete Spur suchen
            if a_id == b_id:
                continue
            gap = fb - last[a_id] - 1
            if gap < 1 or gap > gap_max:
                continue
            boxa = per_id[a_id][last[a_id]]
            (ax, ay), (bx, by) = center(boxa), center(boxb)
            dist = ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5
            if dist <= dist_factor * max(boxa[2], boxb[2]) and (best is None or dist < best_dist):
                best, best_dist = a_id, dist
        if best is not None:                                 # Lücke linear füllen
            la = last[best]
            boxa = per_id[best][la]
            for f in range(la + 1, fb):
                t = (f - la) / (fb - la)
                box = tuple(boxa[i] + t * (boxb[i] - boxa[i]) for i in range(4))
                added[f].append((best, box))
    return [tracked[f] + added[f] for f in range(n)]

## 7. Glätten und Doppelte entfernen (nur offline)

In [ ]:
def smooth_tracks(tracked, k=3, k_back=5):
    n = len(tracked)
    per_id = {}                                              # Kennung -> {Frame: Box}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box

    out = [[] for _ in range(n)]
    for tid, d in per_id.items():
        first = min(d)
        for f in range(max(0, first - k_back), first):       # Anfang nach vorne verlängern
            d[f] = d[first]
        frames = sorted(d)
        arr = np.array([d[f] for f in frames], dtype=float)  # Spalten: x, y, w, h
        cx, cy = arr[:, 0] + arr[:, 2] / 2, arr[:, 1] + arr[:, 3] / 2
        for j, f in enumerate(frames):
            lo, hi = max(0, j - k), min(len(frames), j + k + 1)
            w, h = arr[lo:hi, 2].max(), arr[lo:hi, 3].max()  # größte Größe im Fenster
            mx, my = cx[lo:hi].mean(), cy[lo:hi].mean()      # gemittelte Mitte
            out[f].append((mx - w / 2, my - h / 2, w, h))
    return out


def dedupe(boxes_per_frame, overlap=0.6):
    out = []
    for boxes in boxes_per_frame:
        kept = []
        for b in sorted(boxes, key=lambda b: b[2] * b[3], reverse=True):
            if all(inter_over_smaller(b, kk) < overlap for kk in kept):
                kept.append(b)
        out.append(kept)
    return out

## 8. Verpixeln mit Maske

Alle Boxen (mit Rand) bilden eine Maske; das Bild wird einmal gerastert, nur die Maske wird übernommen.

In [ ]:
def render_mask(input_path, output_path, boxes_per_frame, before=5, after=5, block_px=22,
                padding=0.5, padding_top=0.5, padding_bottom=None):
    cap = cv2.VideoCapture(str(input_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    pb = padding if padding_bottom is None else padding_bottom
    n = len(boxes_per_frame)
    bw, bh = max(1, width // block_px), max(1, height // block_px)    # Zahl der Blöcke im ganzen Bild
    areas = []
    start = time.time()
    try:
        for i in range(n):
            ok, frame = cap.read()
            if not ok:
                break

            # 1. Maske: alle Rechtecke des Zeitfensters in eine Fläche zeichnen
            mask = np.zeros((height, width), dtype=bool)
            for boxes in boxes_per_frame[max(0, i - before):min(n, i + after + 1)]:
                for x, y, w, h in boxes:
                    x0, x1 = int(max(0, x - padding * w)), int(min(width, x + w + padding * w))
                    y0, y1 = int(max(0, y - padding_top * h)), int(min(height, y + h + pb * h))
                    mask[y0:y1, x0:x1] = True

            # 2. Das ganze Bild einmal verpixeln, aber nur die Maske übernehmen
            if mask.any():
                small = cv2.resize(frame, (bw, bh), interpolation=cv2.INTER_AREA)
                pix = cv2.resize(small, (width, height), interpolation=cv2.INTER_NEAREST)
                frame[mask] = pix[mask]

            areas.append(100.0 * mask.mean())
            writer.write(frame)
    finally:
        cap.release()
        writer.release()
    return {"frames": len(areas), "sekunden": round(time.time() - start, 1),
            "flaeche_median_prozent": round(float(np.median(areas)), 1),
            "flaeche_max_prozent": round(float(np.max(areas)), 1)}

## 9. Erkennung für ein ganzes Video (fortsetzbar)

In [ ]:
def detect_video_resumable(input_path, detector, out_file, save_every=500, max_frames=None):
    all_faces = []
    if Path(out_file).exists():
        with open(out_file, "rb") as f:
            all_faces = pickle.load(f)
        print("Fortgesetzt ab Frame", len(all_faces))

    cap = cv2.VideoCapture(str(input_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)
    start = time.time()
    try:
        i = 0
        while i < total:
            ok, frame = cap.read()
            if not ok:
                break
            if i >= len(all_faces):                      # nur neue Frames erkennen
                _, faces = detector.detect(frame)
                all_faces.append(faces)
                if len(all_faces) % save_every == 0:
                    with open(out_file, "wb") as f:
                        pickle.dump(all_faces, f)
                    print(f"{len(all_faces)}/{total} Frames, {time.time() - start:.0f} s")
            i += 1
    finally:
        with open(out_file, "wb") as f:                  # auch bei Abbruch speichern
            pickle.dump(all_faces, f)
        cap.release()
    return all_faces

## 10. Lauf: Erkennung (ca. 2,2 Stunden für VID_76, setzt nach Abbruch fort)

In [ ]:
detector_low = SCRFD(MODEL_SCRFD, input_size=(1280, 736), score_threshold=0.2, pad_value=0)
faces_full = detect_video_resumable(INPUT_VIDEO, detector_low, FACES_FILE)
print("Fertig:", len(faces_full), "Frames")

## 11. Lauf: Tracking, Kontrolle und fertiges Video

In [ ]:
tracked_f = track_with_ids(faces_full, start_threshold=0.5, low_threshold=0.2,
                           max_age=8, near_threshold=0.2)
tracked_f = bridge_gaps(tracked_f, gap_max=40)
boxes_final = dedupe(smooth_tracks(tracked_f, k=3, k_back=5))

# Kontrolle: Abschnitte ohne jede Box (dort ist garantiert nichts verpixelt)
no_box = [i for i, b in enumerate(boxes_final) if len(b) == 0]
runs, start, prev = [], None, None
for i in no_box + [None]:
    if start is None:
        start = prev = i
    elif i is not None and i == prev + 1:
        prev = i
    else:
        if start is not None:
            runs.append((start, prev))
        start = prev = i
print("Frames ohne jede Box:", len(no_box))
print("Längste Abschnitte ohne Box (Frame von-bis, Sekunden):",
      [(a, b, round((b - a + 1) / 25, 1)) for a, b in sorted(runs, key=lambda r: r[1] - r[0], reverse=True)[:5]])

out = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "VID_76_final.mp4"
print(render_mask(INPUT_VIDEO, out, boxes_final, before=0, after=0,
                  padding=0.5, padding_top=0.5, padding_bottom=0.5))

## 12. Zusätzlich: Kopfdetektor (YOLO11s)

**Warum:** Beim Anfall sind Köpfe gedreht, verdeckt oder im Kissen. Der Gesichtsdetektor findet dann oft nichts. Auf dem Ausschnitt Frame 8100–8249 hatten 140 von 150 Frames keine Box. Ein Kopfdetektor sucht den ganzen Kopf und findet ihn in der **linken (RGB) Ansicht** in allen 150 Frames. Die Tests dazu stehen in `Kopfdetektor-YOLO11.ipynb`.

**Grenze:** In der **schwarz-weißen Ansicht** hilft der Kopfdetektor dort nicht. Weder Kontrastanhebung noch andere Bildgrößen noch die Klasse „Person“ des Modells änderten daran etwas. Das bleibt offen.

| | |
|---|---|
| Modell | `models/head_phd/yolov11_phd_s.onnx` (YOLOv11-small, Klasse 1 = Kopf), von Sharath33 auf Hugging Face, trainiert auf CrowdHuman |
| SHA-256 | `3f0a061b37f1398be76fc344c116a8a6c42fb835583cabe33ddd80812ef4af10` |
| Lizenz | laut Seite *openrail++* (Volltext nicht geprüft). CrowdHuman ist ein Forschungsdatensatz. Vor einer Weitergabe klären. |
| Eingabe | fest 640 × 640, Farben **RGB** (das Beispielskript des Autors sagt fälschlich BGR) |
| Ausgabe | 8400 Kandidaten ohne NMS, den NMS machen wir selbst |

Der Detektor liefert wie SCRFD `[x, y, Breite, Höhe, Score]`, damit Tracking und Verpixeln unverändert bleiben.

In [ ]:
MODEL_HEAD = PROJECT_ROOT / "models" / "head_phd" / "yolov11_phd_s.onnx"
HEADS_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_heads_yolo11s.pkl"   # Roh-Erkennungen der Köpfe

class HeadDetector:
    def __init__(self, model_path, score_threshold=0.2, nms_threshold=0.6, class_id=1, rgb=True,
                 size=640, pad_value=114):
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.class_id = class_id          # 0 = Person, 1 = Kopf
        self.rgb = rgb                    # True = RGB (richtig), False = BGR (nur zum Vergleich)
        self.size = size
        self.pad_value = pad_value
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]

        # 1. Letterbox auf size x size
        scale = min(self.size / iw, self.size / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        top, left = (self.size - new_h) // 2, (self.size - new_w) // 2
        canvas = np.full((self.size, self.size, 3), self.pad_value, dtype=np.uint8)
        canvas[top:top + new_h, left:left + new_w] = cv2.resize(img_bgr, (new_w, new_h))

        # 2. Farben und Wertebereich
        if self.rgb:
            canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
        blob = np.ascontiguousarray(canvas.transpose(2, 0, 1)[None]).astype(np.float32) / 255.0

        # 3. Netz
        boxes, scores, classes = self.session.run(None, {self.input_name: blob})
        boxes = boxes[0]                                   # (8400, 4)
        scores = scores[0].reshape(-1)                     # (8400,)
        classes = np.rint(classes[0].reshape(-1)).astype(int)

        # 4. Filtern
        keep = (classes == self.class_id) & (scores >= self.score_threshold)
        if not keep.any():
            return 0, np.empty((0, 5), dtype=np.float32)
        b, sc = boxes[keep], scores[keep]

        # 5. Zurückrechnen: Mitte -> linke obere Ecke, Rand abziehen, durch Faktor teilen
        x = (b[:, 0] - b[:, 2] / 2 - left) / scale
        y = (b[:, 1] - b[:, 3] / 2 - top) / scale
        w, h = b[:, 2] / scale, b[:, 3] / scale
        xywh = np.stack([x, y, w, h], axis=1)

        # 6. NMS
        idxs = np.array(cv2.dnn.NMSBoxes(xywh.tolist(), sc.tolist(), self.score_threshold, self.nms_threshold)).reshape(-1)
        faces = np.array([[*xywh[i], sc[i]] for i in idxs], dtype=np.float32)
        return len(faces), faces

## 13. Kopf-Erkennung für das ganze Video (ca. 25–30 Minuten, setzt nach Abbruch fort)

Dieselbe Funktion `detect_video_resumable` wie bei den Gesichtern. Alles ab Score 0,2 wird gespeichert, die Auswahl kommt im Tracking.

In [ ]:
detector_head = HeadDetector(MODEL_HEAD, score_threshold=0.2)
heads_full = detect_video_resumable(INPUT_VIDEO, detector_head, HEADS_FILE)
print("Fertig:", len(heads_full), "Frames")

## 14. Gesicht und Kopf zusammenführen

Beide Erkennungen laufen **getrennt** durch dieselbe Kette (Tracking → Lücken füllen → Glätten → Doppelte entfernen). Danach:
1. Jede Box bekommt ihren eigenen **Rand**: Gesichtsboxen 0,5 (Stirn, Haare, Kinn), Kopfboxen nur 0,15 (die Kopfbox enthält das schon).
2. Beide Listen werden pro Frame zusammengelegt. Liegt eine Box zu mehr als 60 % in einer größeren, fällt sie weg (`dedupe`).
3. Verpixelt wird die Vereinigung der übrig gebliebenen Rechtecke.

**Startschwelle der Köpfe (`HEAD_START`):** Die Scores eines YOLO-Modells sind anders skaliert als bei SCRFD. Auf dem Ausschnitt waren 0,5 und 0,4 links gleich gut. Mit 0,3 gab es rechts weniger Lücken (62 statt 140 Frames), aber mehr verpixelte Fläche und mehr Risiko für Fehlalarme.

In [ ]:
FACE_PADDING = 0.5
HEAD_PADDING = 0.15
HEAD_START = 0.5
SPLIT = 960      # Grenze zwischen linker (RGB) und rechter (schwarz-weiß) Ansicht in Pixeln
FPS = 25


def chain(faces, start_threshold):
    """Die bekannte Kette: Tracking -> Lücken füllen -> Glätten -> Doppelte entfernen."""
    t = track_with_ids(faces, start_threshold=start_threshold, low_threshold=0.2, max_age=8, near_threshold=0.2)
    t = bridge_gaps(t, gap_max=40)
    return dedupe(smooth_tracks(t, k=3, k_back=5))


def pad_boxes(boxes_per_frame, padding, shape=(1080, 1920)):
    """Weitet jede Box um `padding` x Breite/Höhe nach allen Seiten auf (am Bildrand abgeschnitten)."""
    h, w = shape
    out = []
    for boxes in boxes_per_frame:
        frame_boxes = []
        for x, y, bw, bh in boxes:
            x0, y0 = max(0.0, x - padding * bw), max(0.0, y - padding * bh)
            x1, y1 = min(float(w), x + bw + padding * bw), min(float(h), y + bh + padding * bh)
            frame_boxes.append((x0, y0, x1 - x0, y1 - y0))
        out.append(frame_boxes)
    return out


def in_half(box, half):
    cx = box[0] + box[2] / 2
    return cx < SPLIT if half == "links" else cx >= SPLIT


def area_percent(boxes_per_frame, shape=(1080, 1920)):
    """Anteil des Bildes, der von den (schon aufgeweiteten) Boxen bedeckt wird."""
    h, w = shape
    out = []
    for boxes in boxes_per_frame:
        mask = np.zeros((h, w), dtype=bool)
        for x, y, bw, bh in boxes:
            mask[int(y):int(y + bh), int(x):int(x + bw)] = True
        out.append(100.0 * mask.mean())
    return np.array(out)


n_frames = min(len(faces_full), len(heads_full))
print("Frames: Gesicht", len(faces_full), "| Kopf", len(heads_full), "| verwendet", n_frames)

boxes_faces = pad_boxes(chain(faces_full[:n_frames], 0.5), FACE_PADDING)
boxes_heads = pad_boxes(chain(heads_full[:n_frames], HEAD_START), HEAD_PADDING)
boxes_union = dedupe([f + h for f, h in zip(boxes_faces, boxes_heads)])

### Kontrolle (nur Zahlen)

- **Frames ohne Box:** dort ist nichts verpixelt (Ziel: 0). „links“ und „rechts“ zählen getrennt, ob in der jeweiligen Bildhälfte eine Box ist.
- **Fläche:** Median und Maximum der verpixelten Fläche in Prozent des Bildes.
- Die Vereinigung darf weder links noch rechts schlechter sein als das bessere der beiden Einzelergebnisse.

In [ ]:
def runs_of(flags):
    runs, start = [], None
    for i, f in enumerate(list(flags) + [False]):
        if f and start is None:
            start = i
        if not f and start is not None:
            runs.append((start, i - 1))
            start = None
    return runs


print(f"{'Variante':26s} | ohne Box: gesamt  links  rechts | Fläche Median / Max")
for name, boxes in [("nur Gesicht (10G)", boxes_faces), ("nur Kopf (YOLO11)", boxes_heads), ("Vereinigung", boxes_union)]:
    none_all = sum(1 for b in boxes if not b)
    none_l = sum(1 for b in boxes if not any(in_half(x, "links") for x in b))
    none_r = sum(1 for b in boxes if not any(in_half(x, "rechts") for x in b))
    a = area_percent(boxes)
    print(f"{name:26s} |           {none_all:5d} {none_l:6d} {none_r:7d} | {np.median(a):.1f} % / {a.max():.1f} %")

print()
for half in ("links", "rechts"):
    rl = runs_of([not any(in_half(x, half) for x in b) for b in boxes_union])
    longest = sorted(rl, key=lambda r: r[0] - r[1])[:5]
    print(f"Vereinigung, {half}: längste Abschnitte ohne Box (Frame von-bis, Sekunden):",
          [(a_, b_, round((b_ - a_ + 1) / FPS, 1)) for a_, b_ in longest])

## 15. Fertiges Video mit Gesicht und Kopf

Die Boxen sind schon aufgeweitet, deshalb ist der Rand beim Verpixeln 0. Das bisherige `VID_76_final.mp4` (nur Gesicht) bleibt zum Vergleich erhalten.

In [ ]:
out = OUT_DIR / "VID_76_final_kopf.mp4"
print(render_mask(INPUT_VIDEO, out, boxes_union, before=0, after=0,
                  padding=0, padding_top=0, padding_bottom=0))

## Ergebnis: Eingabegröße 1280 × 736 oder 1920 × 1088?

Vergleich auf den ersten 500 Frames von VID_76 (gleiche Nachbearbeitung, nur Zahlen):

| | 1280 × 736 | 1920 × 1088 |
|---|---|---|
| Frames ohne sicheren Fund (Score ≥ 0,5) | 10 / 500 | 9 / 500 |
| Funde pro Frame (Score ≥ 0,5) | 1,55 | 1,60 |
| Mittlerer Bestwert pro Frame | 0,677 | 0,625 |
| Verpixelte Fläche, Median / Maximum | 5,3 % / 9,0 % | 5,6 % / 7,7 % |
| Erkennungszeit pro Frame | ca. 0,9 s | ca. 2,2 s |

**Entscheidung:** 1280 × 736. Die höhere Auflösung bringt hier keinen Gewinn und kostet rund das 2,5-Fache an Zeit.
Gilt nur für dieses Video (große Gesichter); bei kleinen Gesichtern neu messen.